# Hallucination Detection & Grounding — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/ai-safety/7-2/hallucination-detection.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/ai-safety/7-2/hallucination-detection.ipynb)

> Companion to the **Hallucination Detection & Grounding** article. Build a tiny
> **grounding checker** that splits an answer into claims, checks each one against a
> **source**, and prints a **faithfulness score** — then *watch* it flag a supported
> claim, a hallucinated (unsupported) claim, and an overstated claim, line by line.

## What you'll build

1. A **claim splitter** — break an answer into atomic, checkable claims.
2. A deterministic **mock judge** — token-overlap + polarity/overstatement heuristics
   that label each claim `SUPPORTED` / `CONTRADICTED` / `UNSUPPORTED`.
3. A **faithfulness score** = supported claims / total claims, with the flagged
   claims printed out.

**No API key needed.** The judge is a scripted, deterministic stand-in so the whole
notebook runs top-to-bottom offline. The **last section** shows the *real* LLM-judge
version — it activates only when you set `ANTHROPIC_API_KEY`, and no-ops otherwise.

### Setup (nothing to install for the mock sections)

In [1]:
# Only needed for the FINAL "real LLM judge" section — safe to skip for now:
# %pip install -q anthropic
import os, re
print("ready — no API key required for the mock sections")

ready — no API key required for the mock sections


## Step 1 — the source and the answer

Grounding always compares two things: the **source** (the retrieved context we trust)
and the **answer** (what the model produced). Our job is to check whether every claim
in the answer is backed by the source.

In [2]:
SOURCE = ("In a 2-year trial the drug cut infection risk from 30% to 15%. "
          "The result was not significant. The drug is not yet FDA approved.")

ANSWER = ("The drug cut infection risk by half and it is FDA approved. "
          "The drug produced a large benefit.")

print("SOURCE:\n ", SOURCE)
print("\nANSWER:\n ", ANSWER)

SOURCE:
  In a 2-year trial the drug cut infection risk from 30% to 15%. The result was not significant. The drug is not yet FDA approved.

ANSWER:
  The drug cut infection risk by half and it is FDA approved. The drug produced a large benefit.


## Step 2 — split the answer into atomic claims

A claim is **one** verifiable assertion. "It is FDA approved and works in children"
is really *two* claims — check them separately, because one can be true while the
other is fabricated.

In [3]:
def split_claims(answer: str) -> list:
    """Naive atomic-claim splitter: sentences, then 'and'-joined halves."""
    claims = []
    for sent in re.split(r"(?<=[.!?])\s+", answer.strip()):
        if not sent:
            continue
        for part in re.split(r"\s+and\s+", sent):
            part = part.strip(" .")
            if part:
                claims.append(part)
    return claims

claims = split_claims(ANSWER)
for i, c in enumerate(claims, 1):
    print(f"c{i}: {c}")

c1: The drug cut infection risk by half
c2: it is FDA approved
c3: The drug produced a large benefit


## Step 3 — the mock judge (deterministic, no key)

The judge reads `(claim, source)` and returns a label. It uses three cheap signals:

- **overstatement** — a strong word (*large*, *proven*, *significant*) in the claim
  while the source's evidence is hedged or not significant, so the claim is *stronger
  than the evidence*.
- **contradiction** (intrinsic) — high overlap but opposite polarity (a `not` on one
  side only).
- **support vs unsupported** — how much of the claim's content appears in the source.

This is a teaching stand-in — it misses paraphrase — but the *pipeline* is exactly what
production uses; only the judge changes.

In [4]:
STRONG = {"large", "huge", "significant", "proven", "dramatic", "major", "always", "cured"}
HEDGE  = {"not significant", "no significant", "may", "might", "possibly",
          "not yet", "inconclusive", "unclear"}
NEG    = {"not", "no", "never", "without", "cannot", "n't"}
STOP   = {"the", "a", "is", "it", "of", "by", "in"}

def tokenize(text: str) -> set:
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def negated_words(source: str) -> set:
    """Words the source negates: any token with a negation within 3 tokens BEFORE it,
    scoped to its own sentence. Sentence scoping stops 'not significant. The drug'
    from wrongly negating 'drug' — the 'not' belongs to the previous sentence."""
    neg = set()
    for sentence in re.split(r"[.!?]", source.lower()):
        toks = re.findall(r"[a-z0-9]+", sentence)
        for i, tok in enumerate(toks):
            if NEG & set(toks[max(0, i - 3):i]):
                neg.add(tok)
    return neg

def mock_judge(claim: str, source: str) -> tuple:
    """Return (label, reason). Deterministic stand-in for an LLM judge."""
    src_tokens, claim_tokens = tokenize(source), tokenize(claim)
    content = claim_tokens - NEG - STOP
    shared = content & src_tokens
    overlap = len(shared) / max(1, len(content))
    src_low = source.lower()

    # (a) OVERSTATEMENT: strong claim word + hedged / not-significant source
    if (STRONG & claim_tokens) and any(h in src_low for h in HEDGE):
        return "UNSUPPORTED", "overstated: strong claim but source evidence is hedged/not significant"

    # (b) CONTRADICTION (intrinsic): claim is positive but a shared key term is
    #     negated in the source (claim "FDA approved" vs "not ... approved")
    claim_neg = bool(NEG & claim_tokens)
    if overlap >= 0.6 and not claim_neg and (shared & negated_words(source)):
        return "CONTRADICTED", "polarity mismatch: source negates a key term of the claim"

    # (c) SUPPORTED vs UNSUPPORTED by content overlap
    if overlap >= 0.6:
        return "SUPPORTED", f"source covers {overlap:.0%} of claim terms"
    return "UNSUPPORTED", f"only {overlap:.0%} of claim terms found in source"

# quick sanity check on one claim
print(mock_judge("it is FDA approved", SOURCE))

('CONTRADICTED', 'polarity mismatch: source negates a key term of the claim')


## Step 4 — score the whole answer

Run the judge over every claim, then aggregate:

`faithfulness = supported_claims / total_claims`

Watch three things happen: a **SUPPORTED** claim (backed by the source), a
**CONTRADICTED** claim (the source says the opposite — an *intrinsic* hallucination),
and an **UNSUPPORTED/overstated** claim (stronger than the evidence — *extrinsic*).

In [5]:
def grounding_report(answer: str, source: str) -> dict:
    verdicts = [(c, *mock_judge(c, source)) for c in split_claims(answer)]
    supported = sum(label == "SUPPORTED" for _, label, _ in verdicts)
    n = len(verdicts)
    return {"faithfulness": supported / n if n else 0.0,
            "supported": supported, "n": n, "verdicts": verdicts}

report = grounding_report(ANSWER, SOURCE)
print(f"faithfulness = {report['faithfulness']:.2f}  "
      f"({report['supported']}/{report['n']} claims supported)\n")
for claim, label, reason in report["verdicts"]:
    mark = "OK " if label == "SUPPORTED" else "!! "
    print(f"{mark}[{label:12s}] {claim}")
    print(f"       -> {reason}")

faithfulness = 0.33  (1/3 claims supported)

OK [SUPPORTED   ] The drug cut infection risk by half
       -> source covers 80% of claim terms
!! [CONTRADICTED] it is FDA approved
       -> polarity mismatch: source negates a key term of the claim
!! [UNSUPPORTED ] The drug produced a large benefit
       -> overstated: strong claim but source evidence is hedged/not significant


## Step 5 — gate on the score

In production you don't print the answer — you **gate** it. If faithfulness is below a
threshold, block the answer and fall back to something safe (e.g. quoting the policy).

In [6]:
THRESHOLD = 0.8

def gate(answer: str, source: str, threshold: float = THRESHOLD):
    r = grounding_report(answer, source)
    if r["faithfulness"] >= threshold:
        return {"action": "SEND", "faithfulness": r["faithfulness"]}
    flagged = [c for c, label, _ in r["verdicts"] if label != "SUPPORTED"]
    return {"action": "BLOCK", "faithfulness": r["faithfulness"], "flagged": flagged}

decision = gate(ANSWER, SOURCE)
print(f"decision: {decision['action']}  (faithfulness {decision['faithfulness']:.2f} "
      f"vs threshold {THRESHOLD})")
if decision["action"] == "BLOCK":
    print("blocked because these claims aren't grounded:")
    for c in decision["flagged"]:
        print("   -", c)

# A fully-grounded answer sails through:
good = "The drug cut infection risk from 30% to 15%."
print("\ngrounded answer ->", gate(good, SOURCE))

decision: BLOCK  (faithfulness 0.33 vs threshold 0.8)
blocked because these claims aren't grounded:
   - it is FDA approved
   - The drug produced a large benefit

grounded answer -> {'action': 'SEND', 'faithfulness': 1.0}


## Step 6 — the real LLM judge (optional, needs a key)

Identical pipeline; the deterministic `mock_judge` is swapped for a real model prompted
to be a strict verifier that must **quote the supporting span**. It runs only when
`ANTHROPIC_API_KEY` is set — otherwise it prints a friendly note, so the notebook still
executes top-to-bottom with no key.

In [7]:
def llm_judge(claim: str, source: str):
    if not os.getenv("ANTHROPIC_API_KEY"):
        print("(no ANTHROPIC_API_KEY set — using the mock judge; add a key to run the real one)")
        return mock_judge(claim, source)
    from anthropic import Anthropic
    client = Anthropic()
    resp = client.messages.create(
        model="claude-opus-5", max_tokens=512,
        system=("You are a strict grounding verifier. Given a SOURCE and a CLAIM, decide "
                "whether the source fully supports the claim. Reply with exactly one of "
                "SUPPORTED, CONTRADICTED, or UNSUPPORTED, then quote the supporting span "
                "or say 'no span'. Flag any claim STRONGER than the evidence (e.g. a "
                "'large benefit' when the result is not statistically significant)."),
        messages=[{"role": "user", "content": f"SOURCE:\n{source}\n\nCLAIM:\n{claim}"}])
    text = resp.content[0].text.strip()
    label = next((w for w in ("SUPPORTED", "CONTRADICTED", "UNSUPPORTED") if w in text),
                 "UNSUPPORTED")
    return label, text

print(llm_judge("The drug produced a large benefit.", SOURCE))

(no ANTHROPIC_API_KEY set — using the mock judge; add a key to run the real one)
('UNSUPPORTED', 'overstated: strong claim but source evidence is hedged/not significant')


## Recap

- **Grounding = decompose, check, score, gate.** Split the answer into atomic claims,
  judge each against the source, aggregate into faithfulness, and block below a threshold.
- **Three failure types, three labels:** `SUPPORTED` (point to the span), `CONTRADICTED`
  (intrinsic — opposite of the source), `UNSUPPORTED` (extrinsic — including *overstated*
  claims stronger than the evidence).
- **The judge is swappable.** A token-overlap stand-in teaches the pipeline; production
  swaps in an NLI model or an LLM judge (that quotes its span) with a one-function change.

Back to the **Hallucination Detection & Grounding** article for the full explanation,
diagrams, metrics, and interview Q&A.